In [ ]:
import math
import copy
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.size': 10,
    'figure.dpi': 100
})

np.random.seed(42)

print("Environment configured for Mental Health Risk Prediction Lab")
print("Core algorithm: Gradient Descent for Linear Regression")
print("Application: Predicting mental health risk from social isolation index")

In [ ]:
# COMMUNITY MENTAL HEALTH DATASET
# Social Isolation Index (SII): 0 = well-connected, 10 = extremely isolated
# Mental Health Risk Score (MHRS): 0 = minimal risk, 100 = severe risk

x_train = np.array([1.5, 2.0, 3.0, 4.0, 5.0, 5.5, 6.5, 7.0, 8.5, 9.5])
y_train = np.array([12.0, 18.0, 22.0, 35.0, 40.0, 48.0, 55.0, 62.0, 72.0, 85.0])
participant_ids = [f"PT{i+1:02d}" for i in range(len(x_train))]

print("MENTAL HEALTH SURVEY DATASET")
print("=" * 65)
print(f"{'Participant':<12} {'SII Score':<12} {'MHRS Score':<12} {'Risk Level':<15}")
print("-" * 65)
for i in range(len(x_train)):
    if y_train[i] < 25:
        risk = "Low"
    elif y_train[i] < 50:
        risk = "Moderate"
    elif y_train[i] < 75:
        risk = "Elevated"
    else:
        risk = "High - Refer"
    print(f"{participant_ids[i]:<12} {x_train[i]:<12.1f} {y_train[i]:<12.1f} {risk}")
print("-" * 65)
print(f"Participants: {len(x_train)}")
print(f"SII range: {x_train.min():.1f} - {x_train.max():.1f}")
print(f"MHRS range: {y_train.min():.1f} - {y_train.max():.1f}")
print(f"Correlation coefficient: {np.corrcoef(x_train, y_train)[0,1]:.4f}")

In [ ]:
# VISUALIZATION 1: Comprehensive data overview

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# --- Panel 1: Scatter plot with risk color coding ---
ax1 = fig.add_subplot(gs[0, 0])
risk_colors = []
for y in y_train:
    if y < 25:
        risk_colors.append('#2ecc71')
    elif y < 50:
        risk_colors.append('#f1c40f')
    elif y < 75:
        risk_colors.append('#e67e22')
    else:
        risk_colors.append('#e74c3c')

ax1.scatter(x_train, y_train, c=risk_colors, s=150, edgecolors='black', zorder=5)
for i, pid in enumerate(participant_ids):
    ax1.annotate(pid, (x_train[i], y_train[i]), xytext=(x_train[i]+0.15, y_train[i]+1.5), fontsize=7)
ax1.set_xlabel('Social Isolation Index (0-10)')
ax1.set_ylabel('Mental Health Risk Score (0-100)')
ax1.set_title('SII vs MHRS', fontweight='bold')
ax1.grid(True, alpha=0.3)

# --- Panel 2: SII distribution ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(x_train, bins=5, color='#3498db', edgecolor='black', alpha=0.7)
ax2.axvline(x=np.mean(x_train), color='red', linestyle='--', linewidth=2, label=f'Mean={np.mean(x_train):.1f}')
ax2.set_xlabel('Social Isolation Index')
ax2.set_ylabel('Count')
ax2.set_title('SII Distribution', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# --- Panel 3: MHRS distribution ---
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(y_train, bins=5, color='#9b59b6', edgecolor='black', alpha=0.7)
ax3.axvline(x=np.mean(y_train), color='red', linestyle='--', linewidth=2, label=f'Mean={np.mean(y_train):.1f}')
ax3.set_xlabel('Mental Health Risk Score')
ax3.set_ylabel('Count')
ax3.set_title('MHRS Distribution', fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# --- Panel 4: Risk level pie chart ---
ax4 = fig.add_subplot(gs[1, 0])
risk_counts = [sum(1 for y in y_train if y < 25),
              sum(1 for y in y_train if 25 <= y < 50),
              sum(1 for y in y_train if 50 <= y < 75),
              sum(1 for y in y_train if y >= 75)]
risk_labels = ['Low', 'Moderate', 'Elevated', 'High']
risk_pie_colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
wedges, texts, autotexts = ax4.pie(risk_counts, labels=risk_labels, autopct='%1.0f%%',
                                    colors=risk_pie_colors, startangle=90, textprops={'fontsize': 9})
ax4.set_title('Risk Level Distribution', fontweight='bold')

# --- Panel 5: Ranked bar chart ---
ax5 = fig.add_subplot(gs[1, 1:])
sort_idx = np.argsort(y_train)
sorted_pids = [participant_ids[i] for i in sort_idx]
sorted_mhrs = y_train[sort_idx]
sorted_siis = x_train[sort_idx]
sorted_colors = [risk_colors[i] for i in sort_idx]
bars = ax5.barh(range(len(sorted_pids)), sorted_mhrs, color=sorted_colors, edgecolor='black')
ax5.set_yticks(range(len(sorted_pids)))
ax5.set_yticklabels([f'{pid} (SII={s:.1f})' for pid, s in zip(sorted_pids, sorted_siis)])
ax5.set_xlabel('Mental Health Risk Score')
ax5.set_title('Participants Ranked by Risk Score', fontweight='bold')
ax5.grid(True, alpha=0.3, axis='x')
ax5.axvline(x=50, color='black', linestyle='--', alpha=0.5, label='Clinical threshold (50)')
ax5.legend(fontsize=9)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', edgecolor='black', label='Low risk (< 25)'),
    Patch(facecolor='#f1c40f', edgecolor='black', label='Moderate (25-50)'),
    Patch(facecolor='#e67e22', edgecolor='black', label='Elevated (50-75)'),
    Patch(facecolor='#e74c3c', edgecolor='black', label='High risk (>= 75)'),
]
ax1.legend(handles=legend_elements, fontsize=7, loc='upper left')

plt.suptitle('Mental Health Survey: Comprehensive Data Overview', fontsize=16, fontweight='bold', y=0.98)
plt.show()

print("Top-left: Strong positive trend between isolation and risk score")
print("Top-center/right: Distributions show spread across SII and MHRS ranges")
print("Bottom-left: Risk level proportions in the surveyed population")
print("Bottom: Individual ranking reveals which participants need referral")

In [ ]:
def compute_cost(x, y, w, b):
    """
    Computes the squared-error cost function for linear regression.
    
    In this lab, cost measures total prediction error across all participants.
    Lower cost means the model more accurately predicts mental health risk scores
    from social isolation indices.
    
    Args:
      x (ndarray (m,)): Social Isolation Index values for m participants
      y (ndarray (m,)): Actual Mental Health Risk Scores for m participants
      w (scalar): Slope parameter (risk increase per SII unit)
      b (scalar): Intercept parameter (baseline risk at zero isolation)
    
    Returns:
      total_cost (float): Average squared prediction error
        A cost of 10 means average error of ~4.5 risk-score points
        (sqrt(2 * 10) = 4.47). That could mean the difference between
        'moderate' and 'elevated' risk classification.
    """
    m = x.shape[0]
    cost_sum = 0
    for i in range(m):
        f_wb = w * x[i] + b
        cost_sum += (f_wb - y[i]) ** 2
    total_cost = (1 / (2 * m)) * cost_sum
    return total_cost


# Verify cost function with a rough initial guess
initial_w = 5.0
initial_b = 10.0
initial_cost = compute_cost(x_train, y_train, initial_w, initial_b)
print(f"Initial guess: w={initial_w}, b={initial_b}")
print(f"Cost at initial guess: J = {initial_cost:.2f}")
print(f"Approximate average error: {math.sqrt(2 * initial_cost):.1f} risk-score points")

In [ ]:
def compute_gradient(x, y, w, b):
    """
    Computes the gradient of the cost function for linear regression.
    
    The gradient tells us how to adjust w and b to reduce prediction error.
    This is the core engine that powers gradient descent optimization.
    
    Args:
      x (ndarray (m,)): Social Isolation Index values
      y (ndarray (m,)): Mental Health Risk Scores
      w (scalar): Current slope parameter
      b (scalar): Current intercept parameter
    
    Returns:
      dj_dw (float): Partial derivative of cost w.r.t. w
        Positive = slope too steep, decrease w
        Negative = slope too shallow, increase w
      dj_db (float): Partial derivative of cost w.r.t. b
        Positive = intercept too high, decrease b
        Negative = intercept too low, increase b
    
    Mathematical basis:
      dj_dw = (1/m) * sum( (f_wb - y) * x )
      dj_db = (1/m) * sum( f_wb - y )
      
    The key insight is that errors for HIGH-isolation participants (large x)
    contribute MORE to the w-gradient because they are multiplied by x.
    This means the algorithm prioritizes getting predictions right at
    extreme isolation levels, which is clinically important since those
    patients are at greatest risk.
    """
    m = x.shape[0]
    dj_dw = 0
    dj_db = 0
    
    for i in range(m):
        # Predict risk score using current parameters
        f_wb = w * x[i] + b
        
        # Calculate prediction error for this participant
        error = f_wb - y[i]
        
        # Accumulate gradient components
        # Note: w-gradient is weighted by x[i], so high-isolation participants
        # have proportionally more influence on slope adjustments
        dj_dw_i = error * x[i]
        dj_db_i = error
        
        dj_dw += dj_dw_i
        dj_db += dj_db_i
    
    # Average over all participants
    dj_dw = dj_dw / m
    dj_db = dj_db / m
    
    return dj_dw, dj_db


# Test gradient computation at our initial guess
dj_dw, dj_db = compute_gradient(x_train, y_train, initial_w, initial_b)
print(f"Gradient at w={initial_w}, b={initial_b}:")
print(f"  dj_dw = {dj_dw:.4f} (positive = slope too steep, decrease w)")
print(f"  dj_db = {dj_db:.4f} (positive = intercept too high, decrease b)")
print(f"\nInterpretation: The model should decrease both w and b to reduce cost.")
print(f"The gradient magnitude for w ({abs(dj_dw):.2f}) vs b ({abs(dj_db):.2f})")
print(f"shows w needs a larger relative adjustment due to x-scaling.")

In [ ]:
# VISUALIZATION 2: Gradient visualization - quiver plot and cost surface slices

fig = plt.figure(figsize=(18, 6))

# --- Panel 1: Cost vs w (with b fixed at optimal) and gradient arrows ---
ax1 = fig.add_subplot(131)

# Compute OLS optimal for reference
x_mean, y_mean = np.mean(x_train), np.mean(y_train)
w_opt = np.sum((x_train - x_mean) * (y_train - y_mean)) / np.sum((x_train - x_mean)**2)
b_opt = y_mean - w_opt * x_mean
cost_opt = compute_cost(x_train, y_train, w_opt, b_opt)

w_range = np.linspace(0, 15, 200)
costs_w = [compute_cost(x_train, y_train, w, b_opt) for w in w_range]
ax1.plot(w_range, costs_w, 'b-', linewidth=2)
ax1.plot(w_opt, cost_opt, 'g*', markersize=15, zorder=5, label=f'Optimum w={w_opt:.2f}')

# Draw gradient arrows at three points
test_points_w = [3.0, 7.0, 11.0]
for wp in test_points_w:
    dj_dw, _ = compute_gradient(x_train, y_train, wp, b_opt)
    cp = compute_cost(x_train, y_train, wp, b_opt)
    # Arrow pointing in negative gradient direction (descent direction)
    scale = 0.3
    ax1.annotate('', xy=(wp - scale*dj_dw, cp - scale*abs(dj_dw)*5),
                 xytext=(wp, cp),
                 arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax1.plot(wp, cp, 'ro', markersize=8)
    sign = '+' if dj_dw > 0 else ''
    ax1.annotate(f'grad={sign}{dj_dw:.1f}', (wp, cp), xytext=(wp+0.3, cp+15), fontsize=8, color='red')

ax1.set_xlabel('w (risk sensitivity per SII unit)')
ax1.set_ylabel('Cost J(w, b_opt)')
ax1.set_title('Cost vs w with Gradients', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# --- Panel 2: Quiver plot of gradient field ---
ax2 = fig.add_subplot(132)

w_grid = np.linspace(0, 15, 20)
b_grid = np.linspace(-20, 40, 20)
W_q, B_q = np.meshgrid(w_grid, b_grid)

dW = np.zeros_like(W_q)
dB = np.zeros_like(B_q)
for i in range(W_q.shape[0]):
    for j in range(W_q.shape[1]):
        dw, db = compute_gradient(x_train, y_train, W_q[i,j], B_q[i,j])
        dW[i,j] = dw
        dB[i,j] = db

# Plot cost contour
J_grid = np.zeros_like(W_q)
for i in range(W_q.shape[0]):
    for j in range(W_q.shape[1]):
        J_grid[i,j] = compute_cost(x_train, y_train, W_q[i,j], B_q[i,j])

ax2.contourf(W_q, B_q, J_grid, levels=20, cmap='viridis', alpha=0.6)
ax2.quiver(W_q, B_q, dW, dB, color='red', alpha=0.5, scale=500)
ax2.plot(w_opt, b_opt, 'g*', markersize=15, zorder=5, label='Minimum')
ax2.set_xlabel('w')
ax2.set_ylabel('b')
ax2.set_title('Gradient Field (Quiver Plot)', fontweight='bold')
ax2.legend(fontsize=9)

# --- Panel 3: Per-participant gradient contribution ---
ax3 = fig.add_subplot(133)

contributions_w = []
contributions_b = []
for i in range(len(x_train)):
    f_wb = initial_w * x_train[i] + initial_b
    error = f_wb - y_train[i]
    contributions_w.append(error * x_train[i])
    contributions_b.append(error)

x_pos = np.arange(len(participant_ids))
width = 0.35
bars1 = ax3.bar(x_pos - width/2, contributions_w, width, label='Contribution to dj_dw', color='steelblue', edgecolor='black')
bars2 = ax3.bar(x_pos + width/2, contributions_b, width, label='Contribution to dj_db', color='coral', edgecolor='black')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(participant_ids, rotation=45, fontsize=8)
ax3.axhline(y=0, color='black', linewidth=1)
ax3.set_ylabel('Gradient Contribution')
ax3.set_title('Per-Participant Gradient Contributions', fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3, axis='y')

plt.suptitle('Understanding Gradients: Direction, Field, and Contributions',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Left: Cost curve with gradient arrows. Arrows point downhill toward minimum.")
print("Center: Quiver plot showing gradient field. Arrows point AWAY from minimum")
print("  (gradient descent subtracts the gradient, reversing direction).")
print("Right: Each participant's contribution to the gradient. High-isolation")
print("  participants (PT09, PT10) have outsized influence on the w-gradient.")

In [ ]:
def gradient_descent(x, y, w_in, b_in, alpha, num_iters, cost_function, gradient_function):
    """
    Performs batch gradient descent to optimize parameters w and b.
    
    The algorithm iteratively adjusts w and b by taking steps proportional
    to the negative gradient of the cost function. This is equivalent to
    rolling a ball down the cost surface until it settles at the minimum.
    
    Args:
      x (ndarray (m,)): Input features (Social Isolation Index)
      y (ndarray (m,)): Target values (Mental Health Risk Score)
      w_in (scalar): Initial slope parameter
      b_in (scalar): Initial intercept parameter
      alpha (float): Learning rate (controls step size)
        - Too small: very slow convergence
        - Too large: may diverge (cost increases)
        - Typical range: 0.001 to 1.0 depending on data scale
      num_iters (int): Maximum number of optimization steps
      cost_function: Function to compute cost (for monitoring)
      gradient_function: Function to compute gradients
    
    Returns:
      w (float): Optimized slope parameter
      b (float): Optimized intercept parameter
      J_history (list): Cost at each iteration (for plotting)
      p_history (list): [w, b] at each iteration (for trajectory plotting)
    """
    # Initialize history tracking for visualization
    J_history = []
    p_history = []
    
    # Copy initial parameters (avoid modifying originals)
    b = b_in
    w = w_in
    
    for i in range(num_iters):
        # STEP 1: Compute gradients using CURRENT parameters
        # Both gradients computed BEFORE any updates (simultaneous update rule)
        dj_dw, dj_db = gradient_function(x, y, w, b)
        
        # STEP 2: Update parameters by stepping opposite to gradient
        # The learning rate alpha scales the step size
        # Subtracting the gradient moves us DOWN the cost surface
        temp_w = w - alpha * dj_dw
        temp_b = b - alpha * dj_db
        
        # Assign simultaneously (using temp variables ensures simultaneity)
        w = temp_w
        b = temp_b
        
        # STEP 3: Record cost and parameters for monitoring
        if i < 100000:
            J_history.append(cost_function(x, y, w, b))
            p_history.append([w, b])
        
        # Print progress at regular intervals
        if i % math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:5d}: Cost {J_history[-1]:8.2e} | "
                  f"dj_dw={dj_dw:8.3f}, dj_db={dj_db:8.3f} | "
                  f"w={w:8.4f}, b={b:8.4f}")
    
    return w, b, J_history, p_history

In [ ]:
# RUN GRADIENT DESCENT: Optimize mental health risk model

print("=" * 70)
print("GRADIENT DESCENT: Mental Health Risk Model Optimization")
print("=" * 70)

# Initialize parameters to zero (neutral starting point)
w_init = 0.0
b_init = 0.0

# Learning rate: carefully chosen for this data scale
# SII values range 1.5-9.5, MHRS values range 12-85
# The gradient magnitudes are dominated by the large y-values,
# so a moderate alpha keeps steps reasonable
alpha = 0.01
iterations = 10000

print(f"\nInitial parameters: w={w_init}, b={b_init}")
print(f"Learning rate (alpha): {alpha}")
print(f"Max iterations: {iterations}")
print(f"Initial cost: {compute_cost(x_train, y_train, w_init, b_init):.2f}")
print()

# Run the optimization
w_final, b_final, J_hist, p_hist = gradient_descent(
    x_train, y_train, w_init, b_init, alpha,
    iterations, compute_cost, compute_gradient
)

print(f"\n{'=' * 70}")
print(f"OPTIMIZATION COMPLETE")
print(f"{'=' * 70}")
print(f"Final parameters: w={w_final:.4f}, b={b_final:.4f}")
print(f"Final cost: J={J_hist[-1]:.4f}")
print(f"Analytical optimal: w={w_opt:.4f}, b={b_opt:.4f}, J={cost_opt:.4f}")
print(f"\nModel equation: MHRS = {w_final:.2f} * SII + ({b_final:.2f})")
print(f"Interpretation: Each unit increase in social isolation")
print(f"  corresponds to approximately {w_final:.1f} points higher mental health risk score.")
print(f"Baseline risk at zero isolation: {b_final:.1f} points")

In [ ]:
# VISUALIZATION 3: Convergence diagnostics

fig = plt.figure(figsize=(18, 5))

# --- Panel 1: Cost vs iteration (early phase) ---
ax1 = fig.add_subplot(131)
ax1.plot(range(min(100, len(J_hist))), J_hist[:100], 'b-', linewidth=2)
ax1.set_title('Cost vs Iteration (Early Phase)', fontweight='bold')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Cost J(w,b)')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=cost_opt, color='green', linestyle='--', alpha=0.5, label=f'Optimal J={cost_opt:.2f}')
ax1.legend(fontsize=9)

# --- Panel 2: Cost vs iteration (late phase, log scale) ---
ax2 = fig.add_subplot(132)
start_idx = 1000
ax2.plot(range(start_idx, len(J_hist)), J_hist[start_idx:], 'b-', linewidth=2)
ax2.set_title('Cost vs Iteration (Late Phase)', fontweight='bold')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Cost J(w,b)')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=cost_opt, color='green', linestyle='--', alpha=0.5, label=f'Optimal J={cost_opt:.2f}')
ax2.legend(fontsize=9)

# --- Panel 3: Parameter convergence ---
ax3 = fig.add_subplot(133)
w_history = [p[0] for p in p_hist]
b_history = [p[1] for p in p_hist]
ax3.plot(range(len(w_history)), w_history, 'b-', linewidth=2, label=f'w (target={w_opt:.2f})')
ax3.axhline(y=w_opt, color='blue', linestyle='--', alpha=0.4)
ax3.plot(range(len(b_history)), b_history, 'r-', linewidth=2, label=f'b (target={b_opt:.2f})')
ax3.axhline(y=b_opt, color='red', linestyle='--', alpha=0.4)
ax3.set_title('Parameter Convergence', fontweight='bold')
ax3.set_xlabel('Iteration')
ax3.set_ylabel('Parameter Value')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

plt.suptitle('Gradient Descent Convergence Diagnostics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Left: Rapid cost decrease in first 100 iterations (ball rolling down steep slope)")
print("Center: Gradual approach to minimum in later iterations (flat valley floor)")
print("Right: w and b converge to their optimal values at different rates")
print(f"\nConvergence summary:")
print(f"  Cost decreased from {J_hist[0]:.2f} to {J_hist[-1]:.4f} ({(1-J_hist[-1]/J_hist[0])*100:.1f}% reduction)")
print(f"  Final cost is within {abs(J_hist[-1]-cost_opt):.4f} of analytical minimum")

In [ ]:
# VISUALIZATION 4: Gradient descent trajectory on contour plot

fig = plt.figure(figsize=(16, 6))

# --- Left: Full trajectory on contour ---
ax1 = fig.add_subplot(121)

# Generate dense cost grid
w_vals = np.linspace(-2, 15, 100)
b_vals = np.linspace(-10, 30, 100)
W_grid2, B_grid2 = np.meshgrid(w_vals, b_vals)
J_grid2 = np.zeros_like(W_grid2)
for i in range(W_grid2.shape[0]):
    for j in range(W_grid2.shape[1]):
        J_grid2[i,j] = compute_cost(x_train, y_train, W_grid2[i,j], B_grid2[i,j])

# Contour plot
levels = np.linspace(0, 500, 30)
cf = ax1.contourf(W_grid2, B_grid2, J_grid2, levels=levels, cmap='viridis', alpha=0.6)
ax1.contour(W_grid2, B_grid2, J_grid2, levels=levels, colors='black', linewidths=0.3, alpha=0.3)

# Plot trajectory
traj_w = [p[0] for p in p_hist]
traj_b = [p[1] for p in p_hist]
ax1.plot(traj_w, traj_b, 'r.-', linewidth=1, markersize=2, alpha=0.5, label='GD Trajectory')
ax1.plot(traj_w[0], traj_b[0], 'yo', markersize=10, zorder=5, label='Start')
ax1.plot(w_opt, b_opt, 'g*', markersize=15, zorder=5, label='Optimum')

# Mark every 1000th step with larger marker
for i in range(0, len(traj_w), 1000):
    ax1.plot(traj_w[i], traj_b[i], 'r.', markersize=6)

ax1.set_xlabel('w (slope)')
ax1.set_ylabel('b (intercept)')
ax1.set_title('Full Gradient Descent Trajectory', fontweight='bold')
ax1.legend(fontsize=9)

# --- Right: Zoomed trajectory near minimum ---
ax2 = fig.add_subplot(122)
zoom_w_min, zoom_w_max = w_opt - 2, w_opt + 2
zoom_b_min, zoom_b_max = b_opt - 5, b_opt + 5
ax2.set_xlim(zoom_w_min, zoom_w_max)
ax2.set_ylim(zoom_b_min, zoom_b_max)

# Regenerate contour for zoomed region
w_zoom = np.linspace(zoom_w_min, zoom_w_max, 80)
b_zoom = np.linspace(zoom_b_min, zoom_b_max, 80)
Wz, Bz = np.meshgrid(w_zoom, b_zoom)
Jz = np.zeros_like(Wz)
for i in range(Wz.shape[0]):
    for j in range(Wz.shape[1]):
        Jz[i,j] = compute_cost(x_train, y_train, Wz[i,j], Bz[i,j])

zoom_levels = np.linspace(0, 20, 20)
cf2 = ax2.contourf(Wz, Bz, Jz, levels=zoom_levels, cmap='viridis', alpha=0.6)
ax2.contour(Wz, Bz, Jz, levels=zoom_levels, colors='black', linewidths=0.3, alpha=0.3)

# Plot only late trajectory
late_start = 5000
ax2.plot(traj_w[late_start:], traj_b[late_start:], 'r.-', linewidth=1.5, markersize=3, label='Late GD steps')
ax2.plot(w_opt, b_opt, 'g*', markersize=15, zorder=5, label='Optimum')

ax2.set_xlabel('w')
ax2.set_ylabel('b')
ax2.set_title('Zoomed: Final Convergence Steps', fontweight='bold')
ax2.legend(fontsize=9)

plt.suptitle('Gradient Descent on Cost Surface: Trajectory Visualization',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Left: Full path from (0, 0) to optimal parameters. Large steps initially, small near end.")
print("Right: Zoomed view shows the spiral-like approach to the minimum.")
print("Step size decreases as gradient magnitude shrinks near the minimum.")

In [ ]:
# VISUALIZATION 5: Model fit with prediction errors

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: Data with fitted line ---
ax1 = axes[0]
ax1.scatter(x_train, y_train, c=risk_colors, s=150, edgecolors='black', zorder=5)
for i, pid in enumerate(participant_ids):
    ax1.annotate(pid, (x_train[i], y_train[i]), xytext=(x_train[i]+0.15, y_train[i]+1.5), fontsize=7)

x_line = np.linspace(0, 11, 100)
y_line = w_final * x_line + b_final
ax1.plot(x_line, y_line, 'b-', linewidth=2, label=f'Model: w={w_final:.2f}, b={b_final:.2f}')

# Draw prediction errors
for i in range(len(x_train)):
    pred = w_final * x_train[i] + b_final
    error = pred - y_train[i]
    color = '#e74c3c' if error > 0 else '#2ecc71'
    ax1.plot([x_train[i], x_train[i]], [y_train[i], pred], color=color, linestyle='--', alpha=0.6, linewidth=1.5)

ax1.set_xlabel('Social Isolation Index (0-10)')
ax1.set_ylabel('Mental Health Risk Score (0-100)')
ax1.set_title('Fitted Model with Prediction Errors', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=25, color='#2ecc71', linestyle=':', alpha=0.3, label='_nolegend_')
ax1.axhline(y=50, color='#e67e22', linestyle=':', alpha=0.3, label='_nolegend_')
ax1.axhline(y=75, color='#e74c3c', linestyle=':', alpha=0.3, label='_nolegend_')

# --- Right: Residual plot ---
ax2 = axes[1]
predictions = w_final * x_train + b_final
residuals = y_train - predictions

ax2.scatter(predictions, residuals, c=risk_colors, s=150, edgecolors='black', zorder=5)
ax2.axhline(y=0, color='black', linewidth=1)
ax2.set_xlabel('Predicted MHRS')
ax2.set_ylabel('Residual (Actual - Predicted)')
ax2.set_title('Residual Plot: Model Diagnostics', fontweight='bold')
ax2.grid(True, alpha=0.3)

for i, pid in enumerate(participant_ids):
    ax2.annotate(pid, (predictions[i], residuals[i]), xytext=(predictions[i]+0.5, residuals[i]+0.3), fontsize=7)

plt.suptitle('Optimized Mental Health Risk Prediction Model', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

mae = np.mean(np.abs(residuals))
rmse = np.sqrt(np.mean(residuals**2))
print(f"Model performance:")
print(f"  MAE: {mae:.2f} risk-score points")
print(f"  RMSE: {rmse:.2f} risk-score points")
print(f"  Cost J: {J_hist[-1]:.4f}")
print(f"  R-squared: {1 - np.sum(residuals**2)/np.sum((y_train-y_mean)**2):.4f}")

In [ ]:
# LEARNING RATE COMPARISON: Three scenarios

print("LEARNING RATE ANALYSIS")
print("=" * 70)

alphas = [
    (0.001, 'Too Small (slow convergence)'),
    (0.01, 'Just Right (smooth convergence)'),
    (0.05, 'Large (borderline)'),
    (0.1, 'Too Large (divergence)'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, (alpha_val, desc) in enumerate(alphas):
    ax = axes[idx // 2][idx % 2]
    
    # Run gradient descent with this alpha
    w_t, b_t, J_t, p_t = gradient_descent(
        x_train, y_train, 0.0, 0.0, alpha_val, 500, compute_cost, compute_gradient
    )
    
    # Plot cost history
    if J_t[-1] < 1e10:  # Prevent plotting overflow
        ax.plot(range(len(J_t)), J_t, 'b-', linewidth=2)
        ax.axhline(y=cost_opt, color='green', linestyle='--', alpha=0.5, label=f'Optimal J={cost_opt:.2f}')
    else:
        ax.plot(range(len(J_t)), np.log10(np.array(J_t) + 1), 'r-', linewidth=2)
        ax.set_ylabel('log10(Cost)')
    
    converged = 'CONVERGED' if J_t[-1] < cost_opt * 2 else 'DIVERGED'
    color = 'green' if J_t[-1] < cost_opt * 2 else 'red'
    
    ax.set_title(f'alpha={alpha_val} ({desc})\nFinal J={J_t[-1]:.2f} [{converged}]',
                fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('Iteration')
    if idx < 2:
        ax.set_ylabel('Cost J(w,b)')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    
    print(f"alpha={alpha_val:6.3f} ({desc}): Final J={J_t[-1]:.2f}, w={w_t:.2f}, b={b_t:.2f} [{converged}]")

plt.suptitle('Learning Rate Impact: Convergence vs Divergence', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("\nKey takeaway: alpha=0.01 converges smoothly. alpha=0.1 diverges wildly.")
print("In production ML, monitoring cost for unexpected increase is a critical safeguard.")

In [ ]:
# VISUALIZATION 6: Divergence pathology

# Run with deliberately large alpha to show divergence
print("DEMONSTRATING DIVERGENCE WITH ALPHA=0.1")
print("=" * 60)

w_div, b_div, J_div, p_div = gradient_descent(
    x_train, y_train, 0.0, 0.0, 0.1, 15, compute_cost, compute_gradient
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Panel 1: w trajectory ---
ax1 = axes[0]
w_traj = [p[0] for p in p_div]
ax1.plot(range(len(w_traj)), w_traj, 'ro-', linewidth=2, markersize=6)
ax1.axhline(y=w_opt, color='green', linestyle='--', label=f'Target w={w_opt:.2f}')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('w value')
ax1.set_title('w Oscillation During Divergence', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# --- Panel 2: Cost explosion ---
ax2 = axes[1]
ax2.plot(range(len(J_div)), J_div, 'r-', linewidth=2, marker='o', markersize=4)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Cost J(w,b)')
ax2.set_title('Cost Explosion', fontweight='bold')
ax2.grid(True, alpha=0.3)

# --- Panel 3: Trajectory on contour ---
ax3 = axes[2]
ax3.contourf(W_grid2, B_grid2, J_grid2, levels=np.linspace(0, 500, 30), cmap='viridis', alpha=0.4)
b_traj = [p[1] for p in p_div]
ax3.plot(w_traj, b_traj, 'r.-', linewidth=1.5, markersize=8, label='Divergence path')
ax3.plot(0, 0, 'yo', markersize=10, label='Start')
ax3.plot(w_opt, b_opt, 'g*', markersize=15, label='Target')
ax3.set_xlabel('w')
ax3.set_ylabel('b')
ax3.set_title('Spiraling Away from Minimum', fontweight='bold')
ax3.legend(fontsize=9)

plt.suptitle('Pathology of Divergence: Parameters Spiral Out of Control',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Left: w bounces between increasingly extreme positive and negative values.")
print("Center: Cost grows exponentially rather than shrinking.")
print("Right: The trajectory spirals outward, moving away from the optimum.")
print("\nClinical consequence: A diverged model would predict nonsense risk scores")
print("like -340 or +1200, making the entire screening system useless.")

In [ ]:
# FINAL APPLICATION: Mental health screening tool

print("=" * 70)
print("MENTAL HEALTH RISK SCREENING TOOL")
print("=" * 70)
print(f"Model: MHRS = {w_final:.2f} * SII + ({b_final:.2f})")
print(f"Accuracy: MAE = {mae:.1f} points, R-squared = {1-np.sum(residuals**2)/np.sum((y_train-y_mean)**2):.3f}")
print("=" * 70)

# Screen hypothetical new patients
new_patients = [
    (0.5, 'College student, active social life, clubs and sports'),
    (3.0, 'Working parent, moderate social engagement'),
    (5.0, 'Recently retired, reduced social circle'),
    (7.0, 'Lives alone, remote worker, limited social contact'),
    (9.0, 'Elderly, bereaved, minimal social interaction'),
]

print(f"\n{'Profile':<55} {'SII':<5} {'Pred MHRS':<10} {'Risk Level':<12} {'Action'}")
print("-" * 95)

for sii, profile in new_patients:
    predicted = w_final * sii + b_final
    if predicted < 25:
        risk_level = 'Low'
        action = 'Routine monitoring'
    elif predicted < 50:
        risk_level = 'Moderate'
        action = 'Offer counseling'
    elif predicted < 75:
        risk_level = 'Elevated'
        action = 'Refer to specialist'
    else:
        risk_level = 'High'
        action = 'Urgent intervention'
    print(f"{profile:<55} {sii:<5.1f} {predicted:<10.1f} {risk_level:<12} {action}")

print("-" * 95)
print("\nIMPORTANT ETHICAL CONSIDERATIONS:")
print("1. This model uses SII ONLY. Real screening requires multi-factor assessment.")
print("2. Social isolation is not a choice for many (bereavement, disability, abuse).")
print("3. Model predictions should SUPPORT, not replace, clinical judgment.")
print("4. False negatives (missed high-risk patients) are more harmful than false positives.")
print("5. Data privacy is paramount: mental health data is highly sensitive.")
print("6. Cultural context matters: social norms for isolation vary across communities.")